<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day1/D1_RAG_Assistant_FR_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"></a>
 <span style="color:#6B7B75;">Jour 1 · piste ouverte</span>
 <span style="flex:1;"></span>
 <a href="./D1_RAG_Assistant_EN_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 English</a>
 <a href="./D1_RAG_Assistant_FR.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ piste guidée</a>
</div>

<!-- Construire un assistant RAG sur vos propres publications · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d1_rag_assistant.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;margin-bottom:6px;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">Banque africaine de développement · UA STATAFRIC · STG17 · Jour 1 · 14h45</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  Un assistant RAG sur vos propres publications</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  À la fin de ce laboratoire, vous disposerez d'un assistant qui répond aux questions sur les
  documents de votre office, <b>cite le passage utilisé</b>, et déclare quand la réponse n'y est
  pas. C'est ce dernier comportement qui le rend publiable.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  90 minutes · sans GPU · fonctionne sur portable, sur Colab et sur Kaggle</div>
</div>

> **Ce que vous construisez, en une phrase.** Non pas un modèle qui connaît vos
> statistiques — ce matin a établi qu'aucun modèle ne les connaît. Une chaîne qui
> *récupère* le bon paragraphe dans vos propres documents et demande à un modèle
> de répondre **à partir de ce paragraphe uniquement**.


### Le parcours de ce laboratoire

| # | Étape | Ce que vous apprenez |
|---|-------|----------------------|
| 1 | Installation et fournisseur de modèle | Quels chemins fonctionnent sans clé |
| 2 | Charger un corpus | Pourquoi un PDF scanné est invisible pour votre assistant |
| 3 | Le découper en passages | Ce que changent réellement la taille et le recouvrement |
| 4 | Deux récupérateurs comparés | Mots contre sens, et comment chacun échoue |
| 5 | Le prompt qui interdit l'invention | Où vit la règle de refus |
| 6 | Interroger et lire la citation | Une réponse sans sa source est inutilisable |
| 7 | Le test de refus | Un système qui répond toujours est défaillant |
| 8 | Calibrer le seuil de score | Passer de « répond toujours » à « répond quand il peut » |
| 9 | Évaluer et enregistrer | Distinguer un problème de récupération d'un problème de génération |

**Prérequis.** `scikit-learn`, `pandas` et la boîte à outils `stg17`. En option :
`sentence-transformers` pour le récupérateur sémantique (environ 90 Mo, une fois),
et `pypdf` si votre corpus est en PDF. La cellule suivante installe ce qui manque.

**Un fournisseur de modèle est optionnel pour les étapes 1 à 4 et 8 à 9.** Les
étapes 5 à 7 en exigent un. Sans clé API, le carnet vous indique quels chemins
restent ouverts.


In [ ]:
# La boîte à outils de l'atelier, plus ce qu'exige ce laboratoire. Ré-exécutable.
import subprocess
import sys

REPO = "https://github.com/STG17-Africa/stg17-workshop"

try:
    import stg17  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}.git"], check=False)

from stg17 import env, i18n, rag, ui  # noqa: E402
from stg17.i18n import T  # noqa: E402

S = env.setup({"scikit-learn": "scikit-learn>=1.3", "pandas": "pandas>=2.0"}, lang="FR")
print(S.summary())

---
## 1 · Une variable, et un fournisseur de modèle

`COUNTRY_ISO3` détermine où vos sorties sont écrites et comment elles sont
étiquetées. Mettez votre propre pays et le reste du carnet suit — le registre
couvre les 55 États membres de l'Union africaine.

Le tableau des fournisseurs ci-dessous donne l'image honnête de ce que vous
pouvez exécuter aujourd'hui.


In [ ]:
# Changez cette seule ligne pour exécuter tout le carnet sur votre pays.
COUNTRY_ISO3 = "CIV"

from stg17 import countries, llm  # noqa: E402

C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d1_rag")
print(f"{C.name_en} / {C.name_fr}  ({C.iso3})  ->  {OUT}")

ui.status_table(
    llm.status(),
    title=T("Model providers on this machine", "Fournisseurs de modèles sur cette machine"),
    note=T("You need exactly one. If all are unavailable, steps 1-4 and 8-9 still run: "
           "they are the retrieval half, and retrieval is where most RAG problems live.",
           "Un seul suffit. Si aucun n'est disponible, les étapes 1 à 4 et 8 à 9 "
           "fonctionnent quand même : c'est la moitié « récupération », et c'est là que "
           "vivent la plupart des problèmes du RAG."),
)

---
## 2 · Le corpus

Deux façons de procéder, et la seconde n'est pas un lot de consolation :

**Vos propres documents.** Placez des PDF, des exports Word ou des fichiers texte
dans un dossier et indiquez-le dans `CORPUS_DIR`. C'est l'exercice réel — votre
assistant ne vaut que ce que vous lui donnez.

**Le corpus d'exemple.** Cinq courtes publications fictives dans les formes qu'un
office produit réellement : un rapport de recensement, une note méthodologique,
un bulletin d'enquête, un résumé des dispositions de confidentialité, un
calendrier de diffusion. Chaque chiffre y est inventé et chaque fichier le
déclare dès sa première ligne.

> **Pourquoi l'exemple est délibérément fictif.** Un corpus d'exemple convaincant
> est le chemin le plus court entre un exercice pédagogique et un chiffre inventé
> dans une vraie publication. S'il ressemble à vos propres statistiques,
> quelqu'un finira par le citer.


In [ ]:
# Indiquez un dossier de vos propres documents, ou laissez None pour l'exemple.
CORPUS_DIR = None

if CORPUS_DIR is None:
    CORPUS_DIR = rag.write_sample_corpus(S.path("corpus", "sample"))
    print(T(f"Using the fictional sample corpus at {CORPUS_DIR}",
            f"Corpus d'exemple fictif utilisé : {CORPUS_DIR}"))

docs = rag.load_corpus(CORPUS_DIR)

import pandas as pd  # noqa: E402

ui.result_table(
    pd.DataFrame([{"title": d.title, "words": d.words, "source": d.source} for d in docs]),
    caption=T("Every document your assistant can see. A file missing here is a file "
              "your assistant will never mention — and it will not tell you that.",
              "Tous les documents que votre assistant peut voir. Un fichier absent ici "
              "est un fichier que votre assistant ne mentionnera jamais — et il ne vous "
              "le dira pas."),
)

### L'échec le plus difficile à diagnostiquer

Un PDF scanné est une image. Il n'a pas de couche texte, `load_corpus` n'en
extrait rien, et il est signalé puis ignoré. Sans ce signalement, le symptôme est
un assistant qui ignore paisiblement un rapport présent dans son dossier.

Si vos propres documents sont des scans, ils exigent de l'OCR avant que ce
laboratoire puisse les utiliser — et c'est une discussion du Jour 3, pas du Jour 1.


---
## 3 · Découper les documents en passages

Le modèle ne reçoit jamais un document entier. Il reçoit quelques passages : la
frontière de passage décide donc de ce qu'il peut voir d'un coup.

Deux nombres commandent cela :

- **`size`** — la longueur d'un passage, en caractères. Trop petit, la réponse est
  scindée entre deux passages dont un seul est récupéré. Trop grand, le passage
  porte trois sujets et le récupérateur ne sait plus lequel vous intéressait.
- **`overlap`** — la part du passage précédent répétée au début du suivant. Il
  existe parce que la phrase qui répond à une question est souvent celle qui
  chevauche une frontière.

Exécutez la cellule, puis **changez les nombres et ré-exécutez**. Observez le
nombre de passages et lisez-en un à chaque fois. Il n'y a pas de bonne valeur — il
y a une valeur adaptée à vos documents, et on la trouve en regardant.


In [ ]:
# À FAIRE: Découpez les documents avec rag.chunk_documents, puis affichez le nombre et un passage
...

---
## 4 · Deux récupérateurs, et la manière dont chacun échoue

C'est l'étape qui récompense le plus l'attention.

**TF-IDF** apparie des mots. Il n'exige aucun téléchargement et fonctionne sur
tout réseau. Interrogez-le sur « population » et il trouve les paragraphes
contenant « population ». Demandez-lui « combien de personnes y vivent » et il
peut ne rien trouver, car ces mots n'y figurent pas.

**Les embeddings** apparient du sens. Un modèle transforme chaque passage en
vecteur, et « combien de personnes y vivent » se place près d'un paragraphe sur la
population, même sans mot commun. Coût : un téléchargement unique d'environ 90 Mo.

Ils échouent en sens inverse, et la différence compte :

- TF-IDF ne renvoie **rien** quand les mots ne correspondent pas. Honnête, et facile à détecter.
- Les embeddings renvoient **le passage le plus proche**, toujours — même quand
  rien dans votre corpus n'en approche. Assuré, et invisible sans seuil de score.

La cellule ci-dessous construit les deux et leur pose les mêmes questions.


In [ ]:
# TF-IDF fonctionne toujours. Les embeddings sont tentés, et leur absence est signalée honnêtement.
tfidf = rag.build_index(chunks, kind="tfidf")

try:
    S.installed += env.ensure({"sentence-transformers": "sentence-transformers>=2.2"})
    dense = rag.build_index(chunks, kind="embedding")
except Exception as exc:  # noqa: BLE001
    dense = None
    print(T(f"Embedding retriever unavailable ({type(exc).__name__}). The laboratory "
            f"continues on TF-IDF; the comparison below will show one column only.",
            f"Récupérateur sémantique indisponible ({type(exc).__name__}). Le laboratoire "
            f"continue en TF-IDF ; la comparaison ci-dessous n'aura qu'une colonne."))

In [ ]:
# À FAIRE: Pour chaque question test, affichez le meilleur passage de chaque récupérateur côte à côte
...

> **Lequel utiliser ?** Les embeddings, quand c'est possible. Mais un office sur
> un réseau contraint qui exécute TF-IDF avec un bon jeu de questions obtient un
> assistant qui fonctionne, et un assistant qui fonctionne vaut mieux qu'un
> assistant prévu. Consignez dans votre documentation quel récupérateur a produit
> vos résultats — ils ne sont pas interchangeables, et un lecteur ne peut pas le
> deviner à la sortie.


---
## 5 · Le prompt qui interdit l'invention

La récupération est la moitié du système. L'autre moitié est une instruction
assez stricte pour que le modèle ne se rabatte pas sur sa mémoire quand les
passages sont inutiles.

La règle décisive est la première : si les passages ne contiennent pas la
réponse, répondre par une chaîne de refus fixe et rien d'autre. Une chaîne fixe,
et non une phrase polie, car le carnet doit *détecter* le refus pour le compter.


In [ ]:
# Lisez ceci. C'est court, et chaque ligne y est porteuse.
print(rag.SYSTEM)
print("\n" + "=" * 70)
print(T("The refusal string the notebook looks for:", "La chaîne de refus recherchée :"),
      rag.REFUSAL)

In [ ]:
# Ce que le modèle reçoit réellement — les passages d'abord, la question ensuite.
index = dense if dense is not None else tfidf
preview = rag.build_prompt(PROBES[0], index.search(PROBES[0], k=2))
print(preview[:900] + ("\n...[truncated]" if len(preview) > 900 else ""))

---
## 6 · Interroger, et lire la citation

Les trois étapes s'enchaînent maintenant. Lisez la réponse, puis lisez les
passages en dessous et vérifiez que la réponse s'y trouve réellement.

Faites cette vérification à la main au moins trois fois avant de faire confiance à
la chaîne. C'est le seul moyen d'acquérir le sens de ce qui fonctionne.


In [ ]:
# À FAIRE: Appelez rag.answer() et affichez la réponse avec ses passages
...

---
## 7 · Le test de refus

C'est l'étape que la plupart des premières versions sautent, et c'est elle qui
décide si l'assistant est publiable.

Posez des questions auxquelles votre corpus **ne peut pas** répondre. Un système
correct refuse. Un système incorrect produit une réponse fluide, plausible et
sans source — et il le fera devant un journaliste aussi volontiers qu'ici.


In [ ]:
# Questions sans réponse dans le corpus. Chacune devrait être refusée.
UNANSWERABLE = [
    T("What is the current price of maize per kilogram?",
      "Quel est le prix actuel du maïs au kilogramme ?"),
    T("How many hospitals are there in the northern region?",
      "Combien d'hôpitaux compte la région du nord ?"),
    T("What will the population be in 2050?",
      "Quelle sera la population en 2050 ?"),
]

checks = []
if result is not None:
    for question in UNANSWERABLE:
        a = rag.answer(question, index, k=3)
        checks.append({"question": question,
                       "refused": a.refused,
                       "reply": a.text[:90].replace("\n", " ")})
    ui.result_table(
        pd.DataFrame(checks),
        caption=T("Every row should say refused = True. A False is not a small defect: "
                  "it is the system inventing an answer with a source line attached.",
                  "Chaque ligne devrait indiquer refused = True. Un False n'est pas un "
                  "défaut mineur : c'est le système qui invente une réponse en y "
                  "attachant une ligne de source."),
    )
else:
    print(T("Skipped — needs a model provider.", "Ignoré — exige un fournisseur de modèle."))

> **Si une ligne indique False.** N'en concluez pas que le modèle est mauvais.
> Regardez les passages qu'il a reçus : avec un récupérateur sémantique, il a reçu
> le passage *le plus proche* quelle que soit la distance, et un contenu vaguement
> apparenté est bien plus difficile à refuser qu'un contenu manifestement hors
> sujet. C'est ce que corrige l'étape 8.


---
## 8 · Calibrer le seuil de score

Chaque passage récupéré porte un score de similarité. Les questions auxquelles
votre corpus répond obtiennent un score plus élevé que les autres — mais les deux
plages se chevauchent, et l'endroit où elles se séparent dépend de vos documents.

Trouvez le seuil en mesurant, pas en devinant : notez un ensemble de questions
avec réponse et un ensemble sans réponse, et placez le seuil entre les
distributions.

Trop bas, des passages hors sujet atteignent le modèle, qui peine alors à
refuser. Trop haut, de vraies questions sont refusées. Aucune valeur universelle.


In [ ]:
# À FAIRE: Notez les questions avec et sans réponse, et comparez leurs meilleurs scores
...

---
## 9 · Évaluer, puis conserver les preuves

Deux choses différentes peuvent échouer, et depuis la réponse seule elles se
ressemblent :

- **La récupération a échoué** — le bon paragraphe n'a jamais atteint le modèle.
  Aucun prompt ne corrige cela. Changez le découpage, le récupérateur, ou la
  question.
- **La génération a échoué** — le bon paragraphe est arrivé et la réponse est
  quand même fausse. C'est alors sur le prompt qu'il faut travailler.

`evaluate_retrieval` mesure la première, pour que vous sachiez quelle moitié
corriger.


In [ ]:
# Quel document DEVRAIT répondre à chaque question ? C'est tout le jeu d'évaluation.
CASES = [
    {"question": ANSWERABLE[0], "expect": "labour"},
    {"question": ANSWERABLE[1], "expect": "methodology"},
    {"question": ANSWERABLE[2], "expect": "dissemination"},
    {"question": UNANSWERABLE[0], "expect": None},
]

report = rag.evaluate_retrieval(index, CASES, k=3)
ui.result_table(report, caption=T(
    "hit = the expected document reached the model. A False here means no prompt "
    "change will help.",
    "hit = le document attendu a atteint le modèle. Un False ici signifie qu'aucune "
    "modification du prompt n'y changera rien."))

rate = report["hit"].mean()
print(T(f"Retrieval hit rate: {rate:.0%} on {len(report)} case(s).",
        f"Taux de récupération correcte : {rate:.0%} sur {len(report)} cas."))

In [ ]:
# Le livrable : réponses, passages et provenance, sur disque.
transcript = []
if result is not None:
    for question in ANSWERABLE:
        transcript.append(rag.answer(question, index, k=3, min_score=MIN_SCORE))

path = rag.save_transcript(
    transcript,
    OUT / "rag_transcript.json",
    meta={
        "country": C.iso3,
        "corpus": str(CORPUS_DIR),
        "documents": len(docs),
        "passages": len(chunks),
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "retriever": index.kind,
        "min_score": MIN_SCORE,
        "retrieval_hit_rate": round(float(rate), 3),
    },
)
report.to_csv(OUT / "rag_retrieval_eval.csv", index=False)
print(T(f"Written: {path.name} and rag_retrieval_eval.csv in {OUT}",
        f"Écrits : {path.name} et rag_retrieval_eval.csv dans {OUT}"))

---
### Ce que vous avez

<table>
<tr><td><b>rag_transcript.json</b></td><td>Chaque réponse avec les passages exacts qui la
fondent, le récupérateur utilisé, les paramètres de découpage et le seuil de score.</td></tr>
<tr><td><b>rag_retrieval_eval.csv</b></td><td>Quelles questions ont récupéré le bon document,
et lesquelles non.</td></tr>
</table>

Les deux rejoignent le dépôt de votre pays vendredi. Une réponse sans le passage
dont elle vient ne peut être vérifiée par personne d'autre, et un assistant dont
personne ne peut auditer les sorties n'est pas un assistant qu'un office
statistique peut assumer.

### Les limites de ce que vous avez construit

- Il répond **à partir de vos documents seulement**. Une question sur autre chose
  est correctement refusée, et c'est une qualité.
- La qualité de récupération est bornée par le découpage. Un chiffre scindé entre
  deux passages peut ne jamais être récupéré entier.
- Le seuil de score a été calibré sur une poignée de questions. Vingt seraient
  mieux ; cent seraient défendables.
- **Rien ici ne valide la réponse contre la source.** Il cite un passage ; il ne
  prouve pas que la réponse s'y trouve. Un humain lit encore avant publication.

### Point de contrôle

| Question | Réponse |
|---|---|
| Où vit la règle de refus ? | Dans le prompt système — règle 1 |
| Réponse fausse avec le bon passage : que corrigez-vous ? | Le prompt |
| Réponse fausse avec un passage hors sujet : que corrigez-vous ? | La récupération — découpage, récupérateur ou seuil |
| Pourquoi « absent de mes documents » est-il une bonne réponse ? | Parce que l'alternative est une invention fluide accompagnée d'une citation |


---
## À vous

Quatre extensions, par difficulté croissante. Les deux premières méritent d'être
faites avant vendredi.

1. **Utilisez vos propres documents.** Pointez `CORPUS_DIR` vers de vraies
   publications de votre office et ré-exécutez. Puis rédigez le jeu d'évaluation de
   vingt questions qui compte pour vos utilisateurs, pas pour ce carnet.
2. **Faites bouger la taille des passages.** Divisez-la par deux, multipliez-la par
   deux, et notez le taux de récupération à chaque fois. Mettez les trois nombres
   dans votre dépôt de vendredi — ce tableau est un constat méthodologique.
3. **Rendez la citation cliquable.** Ajoutez un numéro de page à `Chunk` quand la
   source est un PDF, pour qu'une réponse pointe vers une page plutôt que vers un
   indice de passage.
4. **Récupérez deux fois.** Récupérez avec TF-IDF et avec les embeddings, fusionnez
   et dédoublonnez. Mesurez si le taux s'améliore assez pour justifier d'exécuter
   les deux.

Demain à 15h45, cet assistant devient un agent : le modèle cesse de recevoir des
passages que vous avez choisis et se met à choisir quel outil appeler. Tout ce que
vous avez appris sur le refus s'y applique aussi, avec des enjeux plus élevés — une
phrase fausse devient une action fausse.


---
> ### Si quelque chose n'a pas fonctionné
>
> **Aucun fournisseur de modèle.** Les étapes 1 à 4 et 8 à 9 fonctionnent sans, et
> ce sont celles de la récupération — là où vivent la plupart des problèmes du RAG.
> Définissez une clé API avant le Jour 2 et ré-exécutez les étapes 5 à 7.
>
> **Le téléchargement des embeddings a échoué.** TF-IDF est un récupérateur
> légitime, pas un mode dégradé. Notez lequel vous avez utilisé et poursuivez.
>
> **Vos PDF n'ont rien donné.** Ce sont des scans. Ils exigent de l'OCR, hors du
> périmètre de ce laboratoire. Utilisez le corpus d'exemple aujourd'hui et
> apportez la question au Jour 3.
>
> **Tout est refusé.** `MIN_SCORE` est trop élevé. Mettez-le à `0.0` et
> ré-exécutez l'étape 8 avec davantage de questions.
